# 🔬 BioSLATE Project — Phase 5: Expression–Drug Sensitivity Modelling for Synthetic Lethal Targets (GDSC)

**Author:** Faith Ogundimu  
**Date:** July 2025

**Notebook Purpose:**  
This notebook investigates whether **low expression** of synthetic lethal (SL) *target genes*, identified via CRISPR-based screening in HGSOC—is associated with **increased drug sensitivity** in the GDSC pharmacogenomics dataset. This analysis serves as an **orthogonal validation strategy**, mimicking the effect of genetic knockout and providing insight into drug vulnerabilities tied to SL candidates.

> **Context:** SL target genes (e.g. PI4KB, SPAG5, YTHDC1) were identified through statistically rigorous CRISPR screening. Here, we test whether their natural **downregulation** in cancer cell lines correlates with **hypersensitivity** to any drugs, helping prioritise SL targets with therapeutic relevance.

---

### Objectives:

* Load matched GDSC RNA-seq expression data and drug response (AUC) data.
* For each SL target gene:
  * Model AUC ~ expression (linear regression and stratified comparisons).
  * Identify drugs where **low target expression** is significantly associated with **higher sensitivity**.
  * Visualise results via boxplots and regression plots.
* Annotate findings with statistical results and directionality.
* Export key figures and tables for downstream SL reporting and prioritisation.

> This modelling framework helps uncover **functional vulnerabilities** tied to SL target suppression and supports **data-driven drug repurposing** and **therapeutic hypothesis generation**.

__Hits:__  
CDK5 (Representative for Cluster A) | PI4KB  
SCARA3 | SPAG5  
TFB2M | YTHDC1 

### Import Packages

In [1]:
import pandas as pd
import numpy as np

### Clean File Columns and Rows - Expression File

In [25]:
# Load raw Achilles shRNA dataset
expression_df = pd.read_csv('../database_files/DepMap/OmicsExpressionProteinCodingGenesTPMLogp1.csv', index_col=0)  # rows = cell lines, cols = genes

# Clean cell line names — split on '_', keep first part
expression_df.columns = expression_df.columns.str.split(' ').str[0]

# Optional: Drop columns with NA cell line names if any
expression_df = expression_df.loc[:, expression_df.columns.notna()]

expression_df

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,H3C3,CIMIP3,SMIM42,NPBWR1,ACTL10,PANO1,HRURF,PRRC2B,F8A2,F8A1
ACH-001113,4.956577,0.000000,7.577648,3.179411,4.765742,0.037483,1.510547,3.103039,6.697482,4.852304,...,0.093516,0.0,0.0,0.414727,0.077634,0.411901,0.0,5.134808,1.214541,4.315653
ACH-001289,4.954992,0.617243,7.334747,2.783576,3.736280,0.000000,0.149060,3.855960,4.377061,3.466249,...,0.672134,0.0,0.0,0.029840,0.000000,1.026871,0.0,5.951231,0.007227,4.251060
ACH-001339,3.421952,0.000000,7.546069,2.615880,4.476233,0.064571,1.397491,6.833915,3.980336,3.437154,...,0.689369,0.0,0.0,0.127768,0.105594,0.540575,0.0,4.205971,0.030894,2.783448
ACH-001979,4.651643,0.000000,5.946408,2.454515,1.852111,0.000000,7.680796,6.156866,5.042574,3.258451,...,0.000000,0.0,0.0,0.012012,0.000000,0.729221,0.0,4.922916,0.150926,4.661556
ACH-002438,4.336705,0.000000,6.879387,2.262824,3.256491,0.000000,2.253926,6.895436,3.470992,4.308375,...,0.278721,0.0,0.0,0.024008,0.984538,0.549131,0.0,4.949465,0.000000,6.202939
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-002996,4.874320,0.000000,7.691576,2.063810,4.283411,0.000000,5.103099,6.331144,3.700573,3.912123,...,1.108805,0.0,0.0,3.854484,0.838954,0.175788,0.0,3.882936,1.841253,6.631998
ACH-003142,4.484248,0.000000,7.076110,2.319525,4.654034,0.000000,4.831943,6.499084,4.544745,4.258918,...,3.366559,0.0,0.0,0.017747,0.186971,0.743977,0.0,5.733837,0.003743,4.082660
ACH-000323,4.253838,0.000000,6.930962,2.073553,3.591308,0.029919,0.709913,5.920979,3.651373,3.910589,...,1.360915,0.0,0.0,0.000000,0.147617,0.798421,0.0,5.506250,1.317113,2.822339
ACH-000830,3.610157,0.000000,6.466598,2.776647,4.780798,0.362481,0.148231,0.922038,4.035184,5.539438,...,1.786316,0.0,0.0,0.000000,1.792231,0.988758,0.0,7.289263,0.000000,5.712103


### Subset to HGSOC

In [26]:
# Load HGSOC list
hgsoc_lines = pd.read_csv('../database_files/DepMap/cell lines in High-Grade Serous Ovarian Cancer.csv')
hgsoc_lines = hgsoc_lines["Depmap Id"].astype(str).tolist()

# Keep only those present in Expression file
hgsoc_lines_in_expression = [line for line in hgsoc_lines if line in expression_df.index]

# Report missing ones for debugging
missing = set(hgsoc_lines) - set(hgsoc_lines_in_expression)
print(f"⚠️ Missing from Expression File: {missing}")

# Subset rows (cell lines)
expression_hgsoc = expression_df.loc[hgsoc_lines_in_expression]

# Sanity check
print(f'Final matrix shape: {expression_hgsoc.shape}')
expression_hgsoc.to_csv('../results/expression_hgsoc.csv')

# Missing from Expression File: {'ACH-002140' (HEY), 'ACH-002183 (OVMIU)'}
# Cluster Cell Lines - ['ACH-000256', 'ACH-000409', 'ACH-000524', 'ACH-000584']
# No overlap

⚠️ Missing from Expression File: {'ACH-002140', 'ACH-002183'}
Final matrix shape: (21, 19205)


### View and Assess Drug File

In [27]:
# Load raw Achilles shRNA dataset
gdsc_df = pd.read_csv('../database_files/DepMap/sanger-dose-response.csv', index_col=9)

# Optional: Drop columns with NA cell line names if any
gdsc_df = gdsc_df.loc[:, gdsc_df.columns.notna()]

gdsc_df

,DATASET,COSMIC_ID,DRUG_ID,MIN_CONC,MAX_CONC,RMSE_PUBLISHED,Z_SCORE_PUBLISHED,IC50_PUBLISHED,AUC_PUBLISHED,DRUG_NAME,BROAD_ID,upper_limit,ec50,slope,lower_limit,auc,log2.ic50,mse,R2
ARXSPAN_ID,,,,,,,,,,,,,,,,,,,
ACH-002270,GDSC1,683665,1,0.007813,2.0,0.022518,-0.192056,10.977393,0.982116,ERLOTINIB,BRD-K70401845,0.992788,2.839376e+00,-5.670993,0.514389,0.990834,NaN,0.000034,0.904675
ACH-002104,GDSC1,684055,1,0.007813,2.0,0.031831,0.505823,23.133991,0.984820,ERLOTINIB,BRD-K70401845,1.006405,2.864875e-02,-0.186377,0.990054,0.997138,NaN,0.000057,0.028903
ACH-002106,GDSC1,684057,1,0.007813,2.0,0.026047,1.280750,52.935278,0.985696,ERLOTINIB,BRD-K70401845,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ACH-002108,GDSC1,684059,1,0.007813,2.0,0.110056,0.086028,14.774223,0.972701,ERLOTINIB,BRD-K70401845,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ACH-002111,GDSC1,684062,1,0.007813,2.0,0.087010,-0.114395,11.926884,0.944463,ERLOTINIB,BRD-K70401845,0.989580,7.580375e-02,-12.222777,0.894027,0.933185,NaN,0.000623,0.777093
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ACH-000288,GDSC2,905951,2172,0.009766,10.0,0.143863,0.042524,25.410793,0.877741,JQ1,"BRD-K54606188, NA",3.929045,3.287745e+11,-0.012190,-3.449953,0.856099,NaN,0.006178,0.292447
ACH-001065,GDSC2,906862,2172,0.009766,10.0,0.088631,-2.223819,0.339325,0.510590,JQ1,"BRD-K54606188, NA",0.935866,4.096968e-01,-2.014115,0.176515,0.585800,-1.073816,0.003198,0.971991
ACH-000930,GDSC2,907046,2172,0.009766,10.0,0.114748,-0.578942,7.780877,0.843211,JQ1,"BRD-K54606188, NA",0.960799,3.695946e-01,-1.161533,0.687792,0.830671,NaN,0.002792,0.795935


In [28]:
# Load HGSOC list
hgsoc_lines = pd.read_csv('../database_files/DepMap/cell lines in High-Grade Serous Ovarian Cancer.csv')
hgsoc_lines = hgsoc_lines["Depmap Id"].astype(str).tolist()

# Keep only those present in Drug file
hgsoc_lines_in_gdsc = [line for line in hgsoc_lines if line in gdsc_df.index]

# Report missing ones for debugging
missing = set(hgsoc_lines) - set(hgsoc_lines_in_gdsc)
print(f"⚠️ Missing from Drug File: {missing}")

# Subset rows (cell lines)
gdsc_hgsoc = gdsc_df.loc[hgsoc_lines_in_gdsc]

# Sanity check
print(f'Final matrix shape: {gdsc_hgsoc.shape}')
gdsc_hgsoc.to_csv('../results/gdsc_hgsoc.csv')

# Missing from GDSC File: {'ACH-000635', 'ACH-000278', 'ACH-000520', 'ACH-000013', 'ACH-000542', 'ACH-000256', 'ACH-001628', 'ACH-000409'}
# Cluster Cell Lines - ['ACH-000256', 'ACH-000409', 'ACH-000524', 'ACH-000584']
# No overlap

⚠️ Missing from Drug File: {'ACH-000635', 'ACH-000278', 'ACH-000520', 'ACH-000013', 'ACH-000542', 'ACH-000256', 'ACH-001628', 'ACH-000409'}
Final matrix shape: (5544, 19)


### PI4KB Across All Drugs

In [33]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Load cleaned, HGSOC-only data
expression_hgsoc = pd.read_csv('../results/expression_hgsoc.csv', index_col=0)
gdsc_hgsoc = pd.read_csv('../results/gdsc_hgsoc.csv', index_col=0)

# Ensure ARXSPAN_ID is the index for both
expression_hgsoc.index = expression_hgsoc.index.astype(str)
gdsc_hgsoc.index = gdsc_hgsoc.index.astype(str)

# Intersect ARXSPAN IDs across both datasets
shared_ids = gdsc_hgsoc.index.intersection(expression_hgsoc.index)
expression_hgsoc = expression_hgsoc.loc[shared_ids]
gdsc_hgsoc = gdsc_hgsoc.loc[shared_ids]

# Get PI4KB expression
pi4kb_expr = expression_hgsoc['PI4KB']

# Initialise results
results = []

# Loop through drugs
for drug in gdsc_hgsoc['DRUG_NAME'].unique():
    drug_data = gdsc_hgsoc[gdsc_hgsoc['DRUG_NAME'] == drug][['AUC_PUBLISHED']].copy()
    
    # Merge with expression
    merged = drug_data.merge(pi4kb_expr, left_index=True, right_index=True)
    merged = merged.dropna(subset=['AUC_PUBLISHED', 'PI4KB'])

    if len(merged) < 3:
        continue

    # Pearson correlation
    r, p = pearsonr(merged['PI4KB'], merged['AUC_PUBLISHED'])

    results.append({
        'Drug': drug,
        'N_CellLines': merged.index.nunique(),
        'Pearson_r': r,
        'P_value': p,
        'NegCorr': r < 0,
    })

# Compile and save results
results_df = pd.DataFrame(results).sort_values('P_value')
results_df.to_csv('../results/drug_hits/pi4kb_expression_vs_auc_all_drugs.csv', index=False)

# Save top hits
top_hits = results_df[(results_df['NegCorr']) & (results_df['P_value'] < 0.05)]
top_hits.to_csv('../results/drug_hits/pi4kb_hgsoc_top_drug_hits.csv', index=False)

# Print summary
print(top_hits[['Drug', 'N_CellLines', 'Pearson_r', 'P_value']])


                                          Drug  N_CellLines  Pearson_r  \
25                                    XMD14-99           12  -0.841770   
106  VENOTOCLAX, ABT-199, VENECLEXTA, GDC-0199           13  -0.703624   
108                                   CAY10566           13  -0.667406   
352                                      WIKI4           10  -0.740160   
5                                  AVAGACESTAT           13  -0.462135   
325                                       OF-1            7  -0.798314   
13                        AS605240, KIN001-173           12  -0.606418   

      P_value  
25   0.000595  
106  0.007279  
108  0.012691  
352  0.014373  
5    0.022991  
325  0.031371  
13   0.036576  


### SPAG5 Across All Drugs

In [34]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Load cleaned, HGSOC-only data
expression_hgsoc = pd.read_csv('../results/expression_hgsoc.csv', index_col=0)
gdsc_hgsoc = pd.read_csv('../results/gdsc_hgsoc.csv', index_col=0)

# Ensure ARXSPAN_ID is the index for both
expression_hgsoc.index = expression_hgsoc.index.astype(str)
gdsc_hgsoc.index = gdsc_hgsoc.index.astype(str)

# Intersect ARXSPAN IDs across both datasets
shared_ids = gdsc_hgsoc.index.intersection(expression_hgsoc.index)
expression_hgsoc = expression_hgsoc.loc[shared_ids]
gdsc_hgsoc = gdsc_hgsoc.loc[shared_ids]

# Get SPAG5 expression
spag5_expr = expression_hgsoc['SPAG5']

# Initialise results
results = []

# Loop through drugs
for drug in gdsc_hgsoc['DRUG_NAME'].unique():
    drug_data = gdsc_hgsoc[gdsc_hgsoc['DRUG_NAME'] == drug][['AUC_PUBLISHED']].copy()
    
    # Merge with expression
    merged = drug_data.merge(spag5_expr, left_index=True, right_index=True)
    merged = merged.dropna(subset=['AUC_PUBLISHED', 'SPAG5'])

    if len(merged) < 3:
        continue

    # Pearson correlation
    r, p = pearsonr(merged['SPAG5'], merged['AUC_PUBLISHED'])

    results.append({
        'Drug': drug,
        'N_CellLines': merged.index.nunique(),
        'Pearson_r': r,
        'P_value': p,
        'NegCorr': r < 0,
    })

# Compile and save results
results_df = pd.DataFrame(results).sort_values('P_value')
results_df.to_csv('../results/drug_hits/spag5_expression_vs_auc_all_drugs.csv', index=False)

# Save top hits
top_hits = results_df[(results_df['NegCorr']) & (results_df['P_value'] < 0.05)]
top_hits.to_csv('../results/drug_hits/spag5_hgsoc_top_drug_hits.csv', index=False)

# Print summary
print(top_hits[['Drug', 'N_CellLines', 'Pearson_r', 'P_value']])

                                                  Drug  N_CellLines  \
261                                           YK-4-279           10   
167                                          SL 0101-1           12   
346  TELOMERASE INHIBITOR IX, MST-312, MST 312, MST312           10   
264                                             681640           10   
225                                        GEMCITABINE           11   
329                                         BUPARLISIB           10   
3                                           PHENFORMIN           11   
135                                        VINBLASTINE           12   
269                                          SORAFENIB           10   
140                                          TRETINOIN           12   

     Pearson_r   P_value  
261  -0.727353  0.000937  
167  -0.778972  0.002828  
346  -0.824397  0.003346  
264  -0.632847  0.006400  
225  -0.534573  0.012538  
329  -0.742272  0.013951  
3    -0.687434  0.019414  
13

### YTHDC1 Across All Drugs

In [35]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Load cleaned, HGSOC-only data
expression_hgsoc = pd.read_csv('../results/expression_hgsoc.csv', index_col=0)
gdsc_hgsoc = pd.read_csv('../results/gdsc_hgsoc.csv', index_col=0)

# Ensure ARXSPAN_ID is the index for both
expression_hgsoc.index = expression_hgsoc.index.astype(str)
gdsc_hgsoc.index = gdsc_hgsoc.index.astype(str)

# Intersect ARXSPAN IDs across both datasets
shared_ids = gdsc_hgsoc.index.intersection(expression_hgsoc.index)
expression_hgsoc = expression_hgsoc.loc[shared_ids]
gdsc_hgsoc = gdsc_hgsoc.loc[shared_ids]

# Get YTHDC1 expression
ythdc1_expr = expression_hgsoc['YTHDC1']

# Initialise results
results = []

# Loop through drugs
for drug in gdsc_hgsoc['DRUG_NAME'].unique():
    drug_data = gdsc_hgsoc[gdsc_hgsoc['DRUG_NAME'] == drug][['AUC_PUBLISHED']].copy()
    
    # Merge with expression
    merged = drug_data.merge(ythdc1_expr, left_index=True, right_index=True)
    merged = merged.dropna(subset=['AUC_PUBLISHED', 'YTHDC1'])

    if len(merged) < 3:
        continue

    # Pearson correlation
    r, p = pearsonr(merged['YTHDC1'], merged['AUC_PUBLISHED'])

    results.append({
        'Drug': drug,
        'N_CellLines': merged.index.nunique(),
        'Pearson_r': r,
        'P_value': p,
        'NegCorr': r < 0,
    })

# Compile and save results
results_df = pd.DataFrame(results).sort_values('P_value')
results_df.to_csv('../results/drug_hits/ythdc1_expression_vs_auc_all_drugs.csv', index=False)

# Save top hits
top_hits = results_df[(results_df['NegCorr']) & (results_df['P_value'] < 0.05)]
top_hits.to_csv('../results/drug_hits/ythdc1_hgsoc_top_drug_hits.csv', index=False)

# Print summary
print(top_hits[['Drug', 'N_CellLines', 'Pearson_r', 'P_value']])

                Drug  N_CellLines  Pearson_r   P_value
317      VINCRISTINE            7  -0.872739  0.010350
123        FILANESIB           13  -0.606046  0.028124
239         SHIKONIN           10  -0.663216  0.036576
318  PODOPHYLLOTOXIN            7  -0.760603  0.047114


### Visualization

In [39]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import os

# --- Load HGSOC expression and drug data ---
expression_df = pd.read_csv("../results/expression_hgsoc.csv", index_col=0)
gdsc_df = pd.read_csv("../results/gdsc_hgsoc.csv", index_col=0)

# --- Ensure indices are clean strings ---
expression_df.index = expression_df.index.astype(str)
gdsc_df.index = gdsc_df.index.astype(str)

# --- Intersect to get shared cell lines ---
shared_ids = expression_df.index.intersection(gdsc_df.index)
expression_df = expression_df.loc[shared_ids]
gdsc_df = gdsc_df.loc[shared_ids]

# --- Gene to file mapping ---
sl_files = {
    "PI4KB": "../results/drug_hits/pi4kb_hgsoc_top_drug_hits.csv",
    "SPAG5": "../results/drug_hits/spag5_hgsoc_top_drug_hits.csv",
    "YTHDC1": "../results/drug_hits/ythdc1_hgsoc_top_drug_hits.csv"
}

# --- Output directory ---
os.makedirs("../figures/drug_plots/", exist_ok=True)

# --- Generate scatterplots ---
for gene, file in sl_files.items():
    top_hits = pd.read_csv(file)
    
    for _, row in top_hits.iterrows():
        drug = row["Drug"]
        
        # Get AUC values for that drug
        drug_data = gdsc_df[gdsc_df["DRUG_NAME"] == drug][["AUC_PUBLISHED"]].copy()
        expr_data = expression_df[gene]
        
        merged = drug_data.merge(expr_data, left_index=True, right_index=True)
        merged = merged.dropna(subset=["AUC_PUBLISHED", gene])

        if merged.shape[0] < 3:
            continue
        
        # Pearson r/p again just to confirm
        r, p = pearsonr(merged[gene], merged["AUC_PUBLISHED"])
        
        # Plot
        plt.figure(figsize=(6, 5))
        sns.regplot(data=merged, x=gene, y="AUC_PUBLISHED", scatter_kws={'s': 50, 'alpha': 0.8})
        plt.title(f"{drug} vs {gene} Expression\nr = {r:.2f}, p = {p:.3g}")
        plt.xlabel(f"{gene} Expression (log1p TPM)")
        plt.ylabel("AUC (Drug Sensitivity)")
        plt.tight_layout()

        # Save
        fname = f"../figures/drug_plots/{gene}_{drug.replace('/', '_').replace(',', '_')}.png"
        plt.savefig(fname, dpi=300)
        plt.close()

        print(f"Saved: {fname}")

Saved: ../figures/PI4KB_XMD14-99.png
Saved: ../figures/PI4KB_VENOTOCLAX_ ABT-199_ VENECLEXTA_ GDC-0199.png
Saved: ../figures/PI4KB_CAY10566.png
Saved: ../figures/PI4KB_WIKI4.png
Saved: ../figures/PI4KB_AVAGACESTAT.png
Saved: ../figures/PI4KB_OF-1.png
Saved: ../figures/PI4KB_AS605240_ KIN001-173.png
Saved: ../figures/SPAG5_YK-4-279.png
Saved: ../figures/SPAG5_SL 0101-1.png
Saved: ../figures/SPAG5_TELOMERASE INHIBITOR IX_ MST-312_ MST 312_ MST312.png
Saved: ../figures/SPAG5_681640.png
Saved: ../figures/SPAG5_GEMCITABINE.png
Saved: ../figures/SPAG5_BUPARLISIB.png
Saved: ../figures/SPAG5_PHENFORMIN.png
Saved: ../figures/SPAG5_VINBLASTINE.png
Saved: ../figures/SPAG5_SORAFENIB.png
Saved: ../figures/SPAG5_TRETINOIN.png
Saved: ../figures/YTHDC1_VINCRISTINE.png
Saved: ../figures/YTHDC1_FILANESIB.png
Saved: ../figures/YTHDC1_SHIKONIN.png
Saved: ../figures/YTHDC1_PODOPHYLLOTOXIN.png
